# Strategic Investment Banking OS — Transparent Manual Loop 001

This notebook reproduces the first operating-system loop as an open-book process:

1. locate the independent Obsidian vault;
2. inspect the company universe;
3. normalize or verify company `SYN-101`;
4. declare a synthetic +200-basis-point rate shock;
5. calculate every impact-score component visibly;
6. translate the result into candidate banking-opportunity lenses;
7. preview every proposed file mutation;
8. write only after explicit human authorization;
9. validate and preserve an audit trail.

> **Important:** Every company and market assumption is fictional. This is an architecture and governance demonstration—not investment advice, a valuation opinion or a mandate recommendation.

The notebook uses Python's standard library and NumPy. It makes **no LLM call**, consumes **no API tokens** and uses **no pandas**.


## Architecture represented here

| Layer | Notebook responsibility | Vault output |
|---|---|---|
| Memory | Read existing companies and context | `Companies/`, `Data/`, `wiki/` |
| Intake | Normalize SYN-101 | Company record and master CSV |
| Environment | Declare an explicit synthetic shock | `Environment/`, `Scenarios/` |
| Analysis | Expose every score component | Scoring-breakdown CSV |
| Origination | Create separate mandate hypotheses | `Opportunities/` |
| Governance | Require commit approval and backups | `Decisions/`, `Audit/` |
| Continuity | State what enters the next hot cache | Daily brief and audit record |

Calculation and writing are deliberately separated. Nothing is written until the commit gate is opened.


In [ ]:
# 1. Imports and explicit configuration

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from html import escape
import csv, hashlib, io, json, math, os, shutil, tempfile

import numpy as np
from IPython.display import display, HTML, Markdown

VAULT_NAME = "Alejandro-Reynoso-Investment-Banking-Vault"
LOOP_ID = "LOOP-001-COLAB"
RUN_DATE = "2026-07-17"
RATE_SHOCK_BPS = 200

# SAFETY GATE: keep False for the first complete review.
COMMIT_CHANGES = False
BACKUP_BEFORE_WRITE = True

print("COMMIT_CHANGES =", COMMIT_CHANGES)
print("RATE_SHOCK_BPS =", RATE_SHOCK_BPS)


In [ ]:
# 2. Mount Google Drive and resolve the standalone vault

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY_DRIVE = Path("/content/drive/MyDrive")
else:
    MY_DRIVE = Path.cwd()

VAULT = MY_DRIVE / VAULT_NAME
if not VAULT.exists():
    candidates = list(MY_DRIVE.glob(f"**/{VAULT_NAME}"))
    if len(candidates) == 1:
        VAULT = candidates[0]
    elif len(candidates) > 1:
        raise RuntimeError(f"Multiple matching vaults found: {candidates}")
    else:
        raise FileNotFoundError(f"Could not find {VAULT_NAME} under {MY_DRIVE}")

print("Vault:", VAULT)
print("Home note present:", (VAULT / "00 Home.md").exists())


In [ ]:
# 3. Snapshot the vault and load the master universe without pandas

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files_before = sorted(p for p in VAULT.rglob("*") if p.is_file())
manifest_before = {
    str(p.relative_to(VAULT)): {"bytes": p.stat().st_size, "sha256": sha256(p)}
    for p in files_before
}

MASTER_CSV = VAULT / "Data" / "company_master.csv"
NUMERIC_FIELDS = {
    "founded", "employees", "revenue_prev_usd_m", "revenue_usd_m",
    "revenue_growth_pct", "ebitda_usd_m", "ebitda_margin_pct",
    "net_income_usd_m", "free_cash_flow_usd_m", "cash_usd_m", "debt_usd_m",
    "net_debt_usd_m", "enterprise_value_usd_m", "equity_value_usd_m",
    "ev_revenue", "ev_ebitda", "pe_ratio", "net_leverage", "roic_pct",
    "recurring_revenue_pct", "top_customer_concentration_pct", "moat_score",
    "management_score", "execution_risk_score", "esg_score",
}

def parse_number(value):
    if value is None or value == "" or str(value).lower() == "none":
        return None
    return float(value)

with MASTER_CSV.open(encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = list(reader.fieldnames or [])
    records = []
    for raw in reader:
        row = dict(raw)
        for field in NUMERIC_FIELDS:
            if field in row:
                row[field] = parse_number(row[field])
        records.append(row)

print("Files before run:", len(files_before))
print("Company rows loaded:", len(records))
print("Unique IDs:", len({r['id'] for r in records}))


In [ ]:
# 4. Inspect the universe before intake

def display_table(rows, columns, title=None):
    if title:
        display(Markdown(f"### {title}"))
    header = "".join(f"<th style='padding:6px'>{escape(str(c))}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(
            f"<td style='padding:6px;border-top:1px solid #ddd'>{escape(str(row.get(c, '')))}</td>"
            for c in columns
        ) + "</tr>"
        for row in rows
    )
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"))

sector_counts = Counter(r["sector"] for r in records)
profile_counts = Counter(r["investment_style"] for r in records)
display_table([{"Sector": k, "Companies": v} for k, v in sorted(sector_counts.items())], ["Sector", "Companies"], "Sector distribution")
display_table([{"Profile": k, "Companies": v} for k, v in sorted(profile_counts.items())], ["Profile", "Companies"], "Investment-profile distribution")


## Idempotent intake rule

- If `SYN-101` is absent, propose adding it.
- If it is present with the expected identity, verify it and do not duplicate it.
- If ID `SYN-101` belongs to another company, stop immediately.

Repeated analysis must never corrupt institutional memory.


In [ ]:
# 5. Define and validate fictional company SYN-101

SYN101 = {
    "id": "SYN-101", "name": "NexoPort Intelligence", "sector": "Technology",
    "subsector": "Logistics intelligence", "theme": "Supply Chain Resilience",
    "region": "Latin America", "stage": "Startup", "investment_style": "Hypergrowth",
    "ownership": "VC-backed", "founded": 2024, "employees": 185,
    "business_model": "Subscription", "market_position": "Emerging challenger",
    "description": "Fictional AI-assisted logistics-intelligence software company.",
    "revenue_prev_usd_m": 12.5, "revenue_usd_m": 22.5, "revenue_growth_pct": 80.0,
    "ebitda_usd_m": -8.1, "ebitda_margin_pct": -36.0, "net_income_usd_m": -10.4,
    "free_cash_flow_usd_m": -12.7, "cash_usd_m": 32.0, "debt_usd_m": 8.0,
    "net_debt_usd_m": -24.0, "enterprise_value_usd_m": 180.0,
    "equity_value_usd_m": 204.0, "ev_revenue": 8.0, "ev_ebitda": None,
    "pe_ratio": None, "net_leverage": None, "roic_pct": -18.0,
    "recurring_revenue_pct": 88, "top_customer_concentration_pct": 41,
    "moat_score": 7, "management_score": 7, "execution_risk_score": 8,
    "esg_score": 72, "profitability": "Loss-making", "transaction_type": "Growth capital",
    "transaction_rationale": "Raise minority growth capital before financing becomes urgent.",
}

same_id = [r for r in records if r["id"] == "SYN-101"]
if same_id and same_id[0]["name"] != SYN101["name"]:
    raise ValueError("STOP: SYN-101 belongs to another company.")

intake_status = "ALREADY PRESENT — verify only" if same_id else "PROPOSED CREATE"
analysis_records = records if same_id else records + [SYN101]
# Preserve the historical Loop 001 boundary even when this notebook is run
# against the later 102-company vault.
analysis_records = [r for r in analysis_records if r["id"] != "SYN-102"]

ev_reconstructed = SYN101["equity_value_usd_m"] + SYN101["debt_usd_m"] - SYN101["cash_usd_m"]
checks = [
    {"Check": "EV = Equity + Debt - Cash", "Pass": abs(ev_reconstructed - SYN101["enterprise_value_usd_m"]) < 1e-9},
    {"Check": "Unique IDs", "Pass": len({r['id'] for r in analysis_records}) == len(analysis_records)},
    {"Check": "Unique names", "Pass": len({r['name'] for r in analysis_records}) == len(analysis_records)},
    {"Check": "SYN-101 appears once", "Pass": sum(r['id'] == 'SYN-101' for r in analysis_records) == 1},
]
display_table(checks, ["Check", "Pass"], f"Intake validation — {intake_status}")
assert all(c["Pass"] for c in checks)


## Declared synthetic scenario

- Rates rise by 200 basis points.
- Credit underwriting tightens.
- Acquisition debt becomes harder to obtain.
- Loss-making growth companies face a higher cost of new capital.
- Leveraged, cash-flow-negative companies receive earlier refinancing or restructuring review.

The shock does not overwrite company fundamentals. It is a separate analytical layer.


In [ ]:
# 6. Calculate and disclose every impact-score component

RATE_SENSITIVE_SECTORS = {"Real Estate", "Energy and Utilities", "Telecom and Media", "Industrials", "Mobility and Logistics"}
EXCLUDED_SECTORS = {"Financial Services"}  # Require sector-specific balance-sheet analysis.

def impact_score_breakdown(company):
    leverage = company.get("net_leverage")
    if leverage is None or leverage <= 0 or company["sector"] in EXCLUDED_SECTORS:
        return None

    base = 25
    leverage_component = math.log1p(max(0, leverage)) * 10  # Diminishing effect.
    negative_fcf_component = 10 if company["free_cash_flow_usd_m"] < 0 else 0
    low_growth_component = 8 if company["revenue_growth_pct"] < 5 else 0
    sector_component = 6 if company["sector"] in RATE_SENSITIVE_SECTORS else 0
    turnaround_component = 6 if company["investment_style"] == "Turnaround" else 0
    raw_total = base + leverage_component + negative_fcf_component + low_growth_component + sector_component + turnaround_component

    return {
        "id": company["id"], "company": company["name"], "sector": company["sector"],
        "net_leverage": round(leverage, 2), "free_cash_flow_usd_m": company["free_cash_flow_usd_m"],
        "base": base, "leverage_component": round(leverage_component, 2),
        "negative_fcf_component": negative_fcf_component,
        "low_growth_component": low_growth_component, "sector_component": sector_component,
        "turnaround_component": turnaround_component, "raw_total": round(raw_total, 2),
        "final_score": min(95, round(raw_total)),
    }

breakdowns = [b for b in (impact_score_breakdown(r) for r in analysis_records) if b is not None]
order = np.argsort(-np.array([b["final_score"] for b in breakdowns]), kind="stable")
ranked = [breakdowns[i] for i in order]

display_table(ranked[:12], [
    "company", "sector", "net_leverage", "free_cash_flow_usd_m", "base",
    "leverage_component", "negative_fcf_component", "low_growth_component",
    "sector_component", "turnaround_component", "final_score"
], "Complete impact-score decomposition")


## Correct interpretation

The score answers only: **under this synthetic rate shock, which companies deserve earlier banking review?**

It is not a probability, expected return, investment-quality score or mandate-conversion forecast. A high value may indicate distress or vulnerability rather than quality.


In [ ]:
# 7. Translate impact into distinct banking-opportunity lenses

by_name = {r["name"]: r for r in analysis_records}
affected_existing = ranked[:8]
opportunities = [{
    "opportunity_id": "OPP-001", "company": "NexoPort Intelligence",
    "mandate": "Growth capital", "score": 84,
    "why": "Rapid growth and recurring revenue, but negative free cash flow and a higher future cost of capital."
}]

for i, result in enumerate(ranked[:5], start=2):
    company = by_name[result["company"]]
    mandate = "Restructuring or liability management" if company["free_cash_flow_usd_m"] < 0 or company["investment_style"] == "Turnaround" else "Debt refinancing"
    opportunities.append({
        "opportunity_id": f"OPP-{i:03d}", "company": company["name"],
        "mandate": mandate, "score": result["final_score"],
        "why": f"Net leverage {company['net_leverage']:.2f}x; free cash flow ${company['free_cash_flow_usd_m']:,.1f}m."
    })

display_table(opportunities, ["opportunity_id", "company", "mandate", "score", "why"], "Candidate banking lenses")


In [ ]:
# 8. Construct transparent companion artifacts in memory — no writes yet

def csv_text(rows, columns):
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=columns, extrasaction="ignore")
    writer.writeheader(); writer.writerows(rows)
    return buffer.getvalue()

score_columns = ["id", "company", "sector", "net_leverage", "free_cash_flow_usd_m", "base", "leverage_component", "negative_fcf_component", "low_growth_component", "sector_component", "turnaround_component", "raw_total", "final_score"]
pipeline_rows = "\n".join(f"| {o['opportunity_id']} | {o['company']} | {o['mandate']} | {o['score']}/100 | Candidate |" for o in opportunities)
affected_lines = "\n".join(f"- {r['company']}: {r['final_score']}/100" for r in affected_existing)

scenario_md = f'''---
type: scenario-reproduction
scenario_id: SCN-001-COLAB
as_of: {RUN_DATE}
synthetic: true
---
# SCN-001 — Transparent Colab Reproduction

## Declared shock
- Rates: +{RATE_SHOCK_BPS} basis points
- Credit: tighter underwriting and lower leverage tolerance
- External sources: none

## Most affected existing companies
{affected_lines}

The disclosed score is a triage result, not a recommendation.
'''

pipeline_md = f'''---
type: opportunity-pipeline-reproduction
as_of: {RUN_DATE}
loop_id: {LOOP_ID}
synthetic: true
---
# Opportunity Pipeline — Colab Reproduction

| ID | Company | Mandate lens | Score | Status |
|---|---|---|---:|---|
{pipeline_rows}

Every item requires human diligence and approval.
'''

audit_md = f'''---
type: audit-record
audit_id: AUD-001N
date: {RUN_DATE}
loop_id: {LOOP_ID}
---
# AUD-001N — Colab Transparent Reproduction

- Records analyzed: {len(analysis_records)}
- SYN-101 status: {intake_status}
- Shock: +{RATE_SHOCK_BPS} basis points
- Existing companies reviewed: {len(affected_existing)}
- Candidate opportunity lenses: {len(opportunities)}
- External sources: none
- LLM calls: none
- External actions: none
- Human approval required: yes
'''

daily_md = f'''---
type: daily-brief-reproduction
date: {RUN_DATE}
loop_id: {LOOP_ID}
---
# Daily Brief — Colab Transparent Reproduction

SYN-101 was {intake_status.lower()}. The notebook applied the disclosed +{RATE_SHOCK_BPS}-basis-point scenario, reviewed {len(affected_existing)} existing companies and produced {len(opportunities)} candidate banking lenses. Every score component is available in the companion CSV.
'''

relationship_md = '''---
type: relationship-map-reproduction
company: NexoPort Intelligence
loop_id: LOOP-001-COLAB
synthetic: true
---
# NexoPort Intelligence — Colab Relationship Map

- [[Companies/CargoLynx|CargoLynx]] — logistics-software comparable.
- [[Companies/Atlas Freight|Atlas Freight]] — potential commercial partner.
- [[Companies/LuminaGrid|LuminaGrid]] — data-infrastructure comparable.
- [[Companies/HarborChain|HarborChain]] — port-logistics strategic-buyer hypothesis.
- [[Themes/Supply Chain Resilience|Supply Chain Resilience]] — strategic theme.

These are synthetic analytical hypotheses, not actual relationships.
'''

company_md = f'''---
type: company
synthetic: true
company_id: SYN-101
company: NexoPort Intelligence
sector: Technology
subsector: Logistics intelligence
theme: Supply Chain Resilience
stage: Startup
investment_style: Hypergrowth
tags: [company, synthetic-data]
---
# NexoPort Intelligence

Fictional AI-assisted logistics-intelligence software company used for transparent intake testing.

| Metric | Value |
|---|---:|
| Revenue | ${SYN101['revenue_usd_m']:.1f}m |
| Growth | {SYN101['revenue_growth_pct']:.1f}% |
| EBITDA | ${SYN101['ebitda_usd_m']:.1f}m |
| Free cash flow | ${SYN101['free_cash_flow_usd_m']:.1f}m |
| Cash | ${SYN101['cash_usd_m']:.1f}m |
| Debt | ${SYN101['debt_usd_m']:.1f}m |
| Enterprise value | ${SYN101['enterprise_value_usd_m']:.1f}m |

See [[Relationships/NexoPort Intelligence - Colab Reproduction|relationship map]].
'''

planned_writes = {
    Path("Data/loop_001_scoring_breakdown.csv"): csv_text(breakdowns, score_columns),
    Path("Environment/Current Environment - Colab Reproduction.md"): scenario_md,
    Path("Scenarios/SCN-001 - Colab Transparent Reproduction.md"): scenario_md,
    Path("Opportunities/Opportunity Pipeline - Colab Transparent Reproduction.md"): pipeline_md,
    Path("Audit/AUD-001N - Colab Transparent Reproduction.md"): audit_md,
    Path("Daily Briefs/2026-07-17 - Colab Transparent Reproduction.md"): daily_md,
    Path("Relationships/NexoPort Intelligence - Colab Reproduction.md"): relationship_md,
}

for opportunity in opportunities:
    safe_company = opportunity["company"].replace("/", "-")
    safe_mandate = opportunity["mandate"].replace("/", "-")
    card = f'''---
type: opportunity-reproduction
opportunity_id: {opportunity['opportunity_id']}
company: "{opportunity['company']}"
mandate: "{opportunity['mandate']}"
score: {opportunity['score']}
status: Candidate
human_approval: required
synthetic: true
---
# {opportunity['opportunity_id']} — {opportunity['company']}

## Mandate lens
{opportunity['mandate']}

## Why it surfaced
{opportunity['why']}

This is a synthetic candidate requiring human diligence and approval.
'''
    planned_writes[Path(f"Opportunities/{opportunity['opportunity_id']} - {safe_company} - {safe_mandate} - Colab Reproduction.md")] = card

if not same_id:
    planned_writes[Path("Companies/NexoPort Intelligence.md")] = company_md

print("Artifacts in memory:", len(planned_writes))
print("Files written: 0")


In [ ]:
# 9. Preview the exact mutation manifest

mutation_manifest = []
for relative_path, content in planned_writes.items():
    target = VAULT / relative_path
    new_bytes = content.encode("utf-8")
    new_hash = hashlib.sha256(new_bytes).hexdigest()
    if not target.exists():
        action, old_hash = "CREATE", "—"
    else:
        old_hash = sha256(target)
        action = "UNCHANGED" if old_hash == new_hash else "UPDATE"
    mutation_manifest.append({
        "action": action, "path": str(relative_path), "new_bytes": len(new_bytes),
        "old_sha256": old_hash, "new_sha256": new_hash,
    })

display_table(mutation_manifest, ["action", "path", "new_bytes", "old_sha256", "new_sha256"], "Proposed mutations")
print("COMMIT_CHANGES =", COMMIT_CHANGES)


## Human commit gate

First run everything with `COMMIT_CHANGES = False`. Inspect the company record, score decomposition, opportunities and mutation manifest.

Only then change the configuration cell to `COMMIT_CHANGES = True` and rerun. Existing targets are backed up before replacement. This is the notebook equivalent of banker approval before writeback.


In [ ]:
# 10. Commit atomically only when explicitly authorized

def atomic_write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=path.name + ".", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            f.write(content.rstrip() + "\n")
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name): os.unlink(temp_name)

committed = []
if not COMMIT_CHANGES:
    print("DRY RUN COMPLETE — no files were written.")
else:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup_root = VAULT / "Audit" / "Backups" / f"{LOOP_ID}-{timestamp}"
    for relative_path, content in planned_writes.items():
        target = VAULT / relative_path
        existing = target.exists()
        if existing and sha256(target) == hashlib.sha256(content.encode("utf-8")).hexdigest():
            committed.append({"action": "UNCHANGED", "path": str(relative_path)})
            continue
        if existing and BACKUP_BEFORE_WRITE:
            backup = backup_root / relative_path
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, backup)
        atomic_write_text(target, content)
        committed.append({"action": "UPDATE" if existing else "CREATE", "path": str(relative_path)})

    # Append SYN-101 only on a clean baseline. Never duplicate it.
    if not same_id:
        if BACKUP_BEFORE_WRITE:
            backup = backup_root / "Data/company_master.csv"
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(MASTER_CSV, backup)
        with MASTER_CSV.open("a", encoding="utf-8", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore").writerow(SYN101)
        committed.append({"action": "APPEND", "path": "Data/company_master.csv"})

    display_table(committed, ["action", "path"], "Committed changes")
    print("Backup root:", backup_root if BACKUP_BEFORE_WRITE else "Disabled")


In [ ]:
# 11. Validate the model and show the next hot-cache payload

validation = {
    "analysis_company_count": len(analysis_records),
    "unique_ids": len({r['id'] for r in analysis_records}) == len(analysis_records),
    "unique_names": len({r['name'] for r in analysis_records}) == len(analysis_records),
    "syn101_once": sum(r['id'] == 'SYN-101' for r in analysis_records) == 1,
    "affected_existing": len(affected_existing),
    "candidate_opportunities": len(opportunities),
    "score_components_visible": all("final_score" in r and "leverage_component" in r for r in breakdowns),
    "writes_authorized": COMMIT_CHANGES,
}
display_table([{"Validation": k, "Result": v} for k, v in validation.items()], ["Validation", "Result"], "Final validation")
assert validation["unique_ids"] and validation["unique_names"] and validation["syn101_once"]

hot_cache_payload = {
    "loop_id": LOOP_ID,
    "company_universe": len(analysis_records),
    "new_company": "SYN-101 — NexoPort Intelligence",
    "scenario": f"Rates +{RATE_SHOCK_BPS} bps with tighter credit",
    "affected_existing_companies": [r["company"] for r in affected_existing],
    "candidate_opportunities": [f"{o['opportunity_id']} — {o['company']} — {o['mandate']}" for o in opportunities],
    "external_actions": 0,
    "human_approval_required": True,
    "next_test": "Add SYN-102 and use a non-rate scenario before automation.",
}
display(Markdown("### Proposed next-session hot cache"))
print(json.dumps(hot_cache_payload, indent=2, ensure_ascii=False))


## What became transparent

The notebook exposes:

1. the exact vault and dataset used;
2. every field assigned to SYN-101;
3. company-intake invariants and the enterprise-value identity;
4. the environmental assumption;
5. the industry exclusion rule;
6. every numerical score component;
7. the conversion from impact to mandate lens;
8. every proposed file mutation and content hash;
9. the human write gate and backup logic;
10. validation and the context passed to the next session.

The next architectural test should add `SYN-102` under a non-rate scenario. Only after a second coherent loop should the deterministic operations become scripts inside a ChatGPT Skill.
